# NME 3: Predicting NFL Team Points with Machine Learning

During the last NME, we went over how to visualize sports data using Matplotlib and Seaborn. In a typical data science workflow, the next step would be to use what we learned from our data to **make predictions**. 

Today, we'll go over how to build models to make predictions to answer the following question: **Can we predict how many points an NFL team will score in a game?**

We will start with **linear regression**, then build a **decision tree** and a **random forest** to see whether more complicated models can make better predictions. All the models will be built using the [scikit-learn](https://scikit-learn.org/stable/) library.

## Machine Learning Intro
**Machine learning** uses patterns in existing data to make predictions about new data. Today we will use **supervised learning**, meaning we have examples with known outcomes.

- **Linear Regression**: Predicts a numerical value by finding a straight-line relationship between the input features and the target.
- **Decision Tree**: Makes predictions by asking a series of yes-or-no questions about the input features, splitting the data into smaller groups.
- **Random Forest**: Combines predictions from many decision trees to improve accuracy and capture more complex relationships in the data.

## Setup

We will use the [nflreadpy](https://nflreadpy.nflverse.com/api/load_functions/) library to obtain NFL schedules and weekly **team statistics**.

Our features will use **previous games only**, rather than statistics from the game we are trying to predict. This prevents the model from seeing information it would not otherwise have.

In [ ]:
%pip install nflreadpy pyarrow scikit-learn seaborn

With the libraries installed, we can import them into our coding environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nflreadpy as nfl

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

Now we can load game results and weekly team statistics. We use `.to_pandas()` to convert `nflreadpy`'s data to pandas DataFrames.

We'll use the 2021–2024 seasons to train our models and the 2025 season to test the models' predictions.

In [ ]:
seasons = [2021, 2022, 2023, 2024, 2025]
games = nfl.load_schedules(seasons).to_pandas()
team_stats = nfl.load_team_stats(seasons, summary_level="week").to_pandas()

# Keep completed regular-season games
games = games[games["game_type"] == "REG"].dropna(subset=["home_score", "away_score"]).copy()

print(team_stats[["season", "week", "team", "passing_yards", "rushing_yards"]].head())

## Preparing Our Dataset

**Question:** What statistics might help us predict an NFL team's score?

We will use the team's **recent passing yards, rushing yards, and passing attempts**, along with the **opponent's recent yards allowed**, whether the team is at home, and the week of the season.

Each row will represent one team in one game. First, we create rows for the home and away teams and attach their weekly statistics. **We will not use points scored or points allowed as features.**

In [ ]:
# Create one row for each team in each game
home = games[["game_id", "season", "week", "home_team", "away_team", "home_score"]].rename(columns={
    "home_team": "team", "away_team": "opponent", "home_score": "points_scored"
}).copy()
home["home"] = 1

away = games[["game_id", "season", "week", "home_team", "away_team", "away_score"]].rename(columns={
    "away_team": "team", "home_team": "opponent", "away_score": "points_scored"
}).copy()
away["home"] = 0

team_games = pd.concat([home, away], ignore_index=True)

# Match each team's weekly offense to its game
stats = team_stats[["season", "week", "team", "passing_yards", "rushing_yards", "attempts"]].copy()
team_games = team_games.merge(stats, on=["season", "week", "team"], how="inner", validate="one_to_one")
team_games = team_games.sort_values(["season", "week", "game_id"]).reset_index(drop=True)
team_games.head()

Now we'll calculate **three-game averages**. The `.groupby()` function separates each team, while `.shift(1)` excludes the current game and `.rolling(3).mean()` averages the previous three games.

For defense, we calculate how many offensive yards each opponent allowed in its earlier games. That way, every input is available **before kickoff**.

In [ ]:
# Each team's total offensive yards in the completed game
team_games["total_yards"] = team_games["passing_yards"] + team_games["rushing_yards"]

# Add the yards gained by the opposing team in that same game
opponent_yards = team_games[["game_id", "team", "total_yards"]].rename(columns={
    "team": "opponent", "total_yards": "yards_allowed"
})
team_games = team_games.merge(opponent_yards, on=["game_id", "opponent"], how="left", validate="one_to_one")

# Calculate averages using games BEFORE the current one
recent = {
    "passing_yards": "pass_yards_last_3",
    "rushing_yards": "rush_yards_last_3",
    "attempts": "pass_attempts_last_3",
    "yards_allowed": "yards_allowed_last_3"
}
for original, new in recent.items():
    team_games[new] = team_games.groupby(["season", "team"])[original].transform(
        lambda values: values.shift(1).rolling(3).mean()
    )

# Join each opponent's previous defensive performance
opponent_defense = team_games[["game_id", "team", "yards_allowed_last_3"]].rename(columns={
    "team": "opponent", "yards_allowed_last_3": "opp_yards_allowed_last_3"
})
team_games = team_games.merge(opponent_defense, on=["game_id", "opponent"], how="left", validate="one_to_one")

features = ["pass_yards_last_3", "rush_yards_last_3", "pass_attempts_last_3",
            "opp_yards_allowed_last_3", "home", "week"]
model_data = team_games.dropna(subset=features + ["points_scored"]).copy()
model_data[["season", "week", "team", "opponent"] + features + ["points_scored"]].head()

*Pandas note:* `.merge()` joins two tables using matching columns. We use it to attach each opponent's defensive statistics. `.dropna()` removes games without enough earlier games to calculate the three-game averages, including the beginning of each season.

## Splitting Our Data

To find out whether a model can predict new games, we must evaluate it using games that were **not** used to train it.

We will use **2021–2024 for training** and **2025 for testing**. This chronological split lets us train on earlier seasons and evaluate predictions for a later season.

- **Training data:** Used to teach the model.
- **Testing data:** Used to compare predictions with the actual points scored.

In [ ]:
train = model_data[model_data["season"] < 2025]
test = model_data[model_data["season"] == 2025]

X_train, y_train = train[features], train["points_scored"]
X_test, y_test = test[features], test["points_scored"]

print("Training examples:", len(train))
print("Testing examples:", len(test))

## Model 1: Linear Regression

**Linear regression** fits a straight-line relationship between our features and points scored. It gives us a simple model to compare against more complicated methods. We use `.fit()` to train it and `.predict()` to make predictions.

We will compare models using **mean absolute error (MAE)**, the average number of points that our predictions differ from the actual scores. **Lower MAE is better.**

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)
linear_mae = mean_absolute_error(y_test, linear_pred)
print("Linear Regression MAE:", round(linear_mae, 2))

## Model 2: Decision Tree

A **decision tree** learns a series of questions about our features, such as whether a team's recent rushing average is above a certain value. Each path ends with a predicted score.

We use `DecisionTreeRegressor()` to create the model. The `max_depth` argument limits how many levels of questions it can ask. A deeper tree can learn more complicated patterns, but it may also **overfit** the training data.

In [ ]:
tree_model = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)
tree_mae = mean_absolute_error(y_test, tree_pred)
print("Decision Tree MAE:", round(tree_mae, 2))

We can also use Matplotlib to visualize the decision tree.

In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(tree_model, feature_names=features, filled=True, rounded=True, max_depth=2, fontsize=8)
plt.show()

## Model 3: Random Forest

**Question:** Could combining many decision trees give us more accurate predictions?

A **random forest** builds many decision trees using different samples of the data and different subsets of features. It averages their predictions, which often makes it more stable than a single decision tree.

We use `RandomForestRegressor()` to create the model. `n_estimators` controls the number of trees and `max_depth` controls how complex each tree can become.

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    random_state=42
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)

print("Random Forest MAE:", round(rf_mae, 2))

## Comparing Our Models

**Question:** Which model made the most accurate predictions on 2025 games?

We can use `mean_absolute_error()` and a bar chart to compare the three models. Remember that **a more complicated model is not guaranteed to perform better**.

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree", "Random Forest"],
    "MAE": [linear_mae, tree_mae, rf_mae]
})

sns.barplot(data=results, x="Model", y="MAE")
plt.title("Model Comparison on 2025 Test Games")
plt.ylabel("Mean Absolute Error (Points)")
plt.xlabel("")
plt.show()

print(results.round(2))

### Understanding Feature Importance

**Question:** Which features does our Random Forest rely on most?

We can use `.feature_importances_` to see how much each feature contributes to the trees' predictions. These values describe how the model works; **they do not prove that a feature causes teams to score points**.

In [ ]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

sns.barplot(data=importance, x="Importance", y="Feature")
plt.title("Random Forest Feature Importance")
plt.show()

We can also compare the Random Forest's predicted points with the actual scores. If predictions were perfect, every point would be on the dashed diagonal line.

In [ ]:
sns.scatterplot(x=y_test, y=rf_pred, alpha=0.55)
plt.plot([0, 60], [0, 60], linestyle="--", color="gray")
plt.title("Random Forest: Actual vs. Predicted Points (2025)")
plt.xlabel("Actual Points Scored")
plt.ylabel("Predicted Points Scored")
plt.show()

## Try It Yourself: Tune the Models

**Question:** Can you change the models' **hyperparameters** to predict points more accurately?

**Hyperparameters** are settings we choose *before* training a model. Create your own models from scratch in the empty cell below:

1. Train a new **decision tree**, trying different values of `max_depth` (for example, 2, 5, or 10) and `min_samples_leaf`.
2. Train a new **random forest**, trying different `n_estimators`, `max_depth`, or `min_samples_leaf`.
3. Predict points using `X_test` and calculate each model's **MAE**.
4. Can you beat the original decision tree or random forest? Can either beat linear regression?

*Hint:* Look back at `.fit()`, `.predict()`, and `mean_absolute_error()`. We are using 2025 to compare settings for this introductory exercise. Since we are testing multiple choices on the same games, the best MAE should not be treated as an unbiased estimate of future performance.

In [ ]:
# INSERT CODE HERE


## Wrap-up

| Concept | What it does | Python function / attribute |
|---|---|---|
| Features and target | Defines inputs and what we predict | `X` and `y` |
| Training | Learns patterns from the training data | `.fit()` |
| Predictions | Predicts points in new games | `.predict()` |
| Linear regression | Creates a simple regression baseline | `LinearRegression()` |
| Decision tree | Predicts using a series of feature-based decisions | `DecisionTreeRegressor()` |
| Random forest | Averages predictions across multiple trees | `RandomForestRegressor()` |
| Hyperparameters | Control model complexity | `max_depth`, `n_estimators`, `min_samples_leaf` |
| Model evaluation | Measures average prediction error | `mean_absolute_error()` |